# Unidade 5 — Guia de estudo

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/00_guia_da_unidade.ipynb)
## Como comparar grupos, períodos e documentos?

![Dois conjuntos de documentos passam por distribuições e incerteza; dois textos passam por conjuntos, vetores e alinhamento; ambos os caminhos retornam a documentos inspecionados por uma lupa.](https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/imagens/00_abertura_conceitual.png?raw=1)

*Ilustração conceitual gerada para a abertura. Os documentos são genéricos e
fictícios; a imagem não constitui evidência histórica.*

**Problema orientador:** como verificar se grupos sociais, instituições,
períodos ou documentos apresentam diferenças relevantes?

A Unidade 4 descreveu padrões e formulou hipóteses exploratórias. Agora
aprenderemos a definir uma comparação, estimar seu tamanho, explicitar a
incerteza quando pertinente e comparar documentos por representações textuais.
Comparar não é apenas subtrair números ou ordenar similaridades: é justificar
unidades, medidas, pressupostos e alcance.

In [1]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_05'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Ambiente preparado em: /content/disciplina_computacao_aplicada_humanidades_digitais/unidade_05


A pergunta orientadora se divide em dois percursos que compartilham a mesma
exigência: toda medida precisa retornar aos casos e ao contexto.

## Objetivos e percurso

![Seis etapas ligam pergunta, estimativa, incerteza, representação textual, comparação e interpretação; uma seta retorna às decisões anteriores.](https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/imagens/00_percurso_comparacao.svg?raw=1)

| Notebook | Pergunta central | Produto |
|---|---|---|
| 00 | O que torna uma comparação defensável? | Escolha preliminar |
| 01 | Qual é o tamanho da diferença? | Quadro de estimativas |
| 02 | Que incerteza acompanha a estimativa? | Ficha inferencial |
| 03 | Como transformar documentos em vetores? | Matriz textual |
| 04 | Em que sentido dois textos são semelhantes? | Relatório de pares |
| 05 | Que comparação sustenta o argumento? | Análise comparativa |

Ao final, você deverá distinguir estimativa, incerteza, valor de *p*, tamanho
de efeito e relevância; construir matriz documento-termo, TF-IDF e três medidas
de comparação textual; e inspecionar qualitativamente os casos.

In [2]:
import json
import pandas as pd

documentos = pd.read_csv("dados/documentos.csv")
proveniencia = json.loads(open("dados/proveniencia.json", encoding="utf-8").read())
print("Registros:", len(documentos))
print("Natureza:", proveniencia["natureza"])
documentos.head(3)

Registros: 24
Natureza: dados inteiramente fictícios


,id_documento,ano,genero,local,tema,palavras,pessoas,paginas,texto
0,D001,1890,editorial,Capital,educação,320,0,1,A escola noturna amplia a instrução pública e ...
1,D002,1891,notícia,Capital,trabalho,367,3,2,O trabalho nas oficinas reúne jornada salário ...
2,D003,1892,carta,Capital,progresso,414,6,3,A estrada e a máquina anunciam progresso e tam...


A inspeção confirma que trabalharemos com os mesmos 24 registros fictícios da
Unidade 4. Essa continuidade permite testar hipóteses já formuladas, mas não
transforma a coleção em amostra probabilística.

## Regra de interpretação

Para toda comparação, registre:

1. unidades, grupos e medida;
2. diferença ou similaridade observada;
3. incerteza ou sensibilidade pertinente;
4. casos que qualificam o agregado;
5. afirmação sustentada e limite.

Intervalos e testes só possuem interpretação inferencial sob um procedimento e
uma população-alvo defensáveis. Em um corpus completo, a diferença descritiva
pode ser o resultado relevante; em uma coleção de conveniência, um pequeno valor
de *p* não repara o mecanismo de seleção.

## Questões críticas

Elas atravessarão todos os experimentos:

1. O que significa considerar dois grupos diferentes?
2. O que significa considerar dois textos semelhantes?
3. Como a escolha da medida altera a conclusão?
4. Uma diferença estatística é necessariamente relevante?

## U05-A01 — Diagnóstico inicial

**Atividade individual — 10 minutos.** Responda em Markdown:

- Quando uma diferença de médias é historicamente importante?
- Um intervalo de confiança contém 95% dos dados?
- O valor de *p* informa a probabilidade de a hipótese nula ser verdadeira?
- Dois textos com as mesmas palavras são necessariamente equivalentes?
- Que hipótese da Unidade 4 você gostaria de comparar?

**Minha resposta**

| Questão | Resposta |
|---|---|
| Quando uma diferença de médias é historicamente importante? | Quando seu tamanho é grande o bastante para mudar uma leitura do período — não basta ser diferente de zero. No projeto, a taxa de "liberdade" em 1985 (4,92/mil) e a de 2003 (0,00/mil) diferem de um jeito que já importa antes de qualquer teste: 12 palavras a menos em Lula 2003 não mudariam essa leitura. |
| Um intervalo de confiança contém 95% dos dados? | Não. Um IC de 95% é um procedimento que, repetido em amostras da mesma população, captura o parâmetro verdadeiro em 95% das vezes — não é um intervalo que abrange 95% das observações. Os dados brutos podem estar todos fora dele. |
| O valor de *p* informa a probabilidade de a hipótese nula ser verdadeira? | Não. É a probabilidade de observar um resultado ao menos tão extremo quanto o obtido, *assumindo* que a hipótese nula é verdadeira. Não diz nada sobre a probabilidade da própria hipótese. |
| Dois textos com as mesmas palavras são necessariamente equivalentes? | Não. `liberdade de oprimir o vulnerável` (Lula, 2023) e `liberdade é um legado de Deus` (Sarney, 1985) compartilham o alvo lexical e nada do sentido — um usa o termo para recusar o significado que o adversário lhe dá, o outro para afirmá-lo como valor. Bag-of-words e TF-IDF ignoram essa diferença; é por isso que o retorno ao trecho continua sendo necessário mesmo depois de qualquer medida vetorial. |
| Que hipótese da Unidade 4 eu gostaria de comparar? | H2: a taxa de "liberdade" por mil palavras forma um U ao longo de 1985–2023, com valores altos nas pontas (1985 e 2019–2023) e um vale entre 1999 e 2007. Quero comparar o grupo "pontas" (1985, 1990, 1995, 2019, 2023) contra o grupo "centro" (1999, 2003, 2007, 2011, 2015, 2016) e estimar o tamanho dessa diferença — sabendo de antemão que, com 12 discursos completos (não uma amostra de uma população maior), um teste de significância não teria muito o que acrescentar à diferença descritiva. 

O diagnóstico torna visíveis pressupostos que serão revistos. O produto final
deverá mostrar uma escolha métrica, uma análise de sensibilidade e retorno aos
casos, não apenas uma saída computacional.

## Produto e avaliação

A análise comparativa será avaliada por alinhamento à pergunta, comparabilidade
dos grupos, correção da medida, transparência dos pressupostos, inspeção de casos,
relevância substantiva, limites e reprodutibilidade.

Siga para o Notebook 01. Começaremos pelo tamanho observado da diferença antes de
introduzir testes ou intervalos.